#Github

In [1]:
!git clone https://github.com/FaridRash/brain-ct-hemorrhage-segmentation

Cloning into 'brain-ct-hemorrhage-segmentation'...
remote: Enumerating objects: 239, done.
remote: Counting objects: 100% (70/70), done.
remote: Compressing objects: 100% (68/68), done.
remote: Total 239 (delta 15), reused 5 (delta 0), pack-reused 169 (from 2)
Receiving objects: 100% (239/239), 572.77 MiB | 18.98 MiB/s, done.
Resolving deltas: 100% (79/79), done.
Updating files: 100% (160/160), done.


#Libraries

In [2]:
import os
import numpy as np
import nibabel as nib
import pandas as pd
from sklearn.model_selection import train_test_split



#Check the masks

In [ ]:
# paths
ct_dir = "/content/brain-ct-hemorrhage-segmentation/Data/computed-tomography-images-for-intracranial-hemorrhage-detection-and-segmentation-1.3.1/ct_scans"
mask_dir = "/content/brain-ct-hemorrhage-segmentation/Data/computed-tomography-images-for-intracranial-hemorrhage-detection-and-segmentation-1.3.1/masks"

output_dir = "/content/splits"
os.makedirs(output_dir, exist_ok=True)

scan_ids = []
labels = []

# ---- STEP 1: build scan-level labels ----
for file_name in sorted(os.listdir(mask_dir)):

    if not file_name.endswith(".nii"):
        continue

    mask_path = os.path.join(mask_dir, file_name)

    mask = nib.load(mask_path).get_fdata()

    # if ANY voxel > 0 → hemorrhage
    label = 1 if np.any(mask > 0) else 0

    scan_id = file_name.replace(".nii", "")

    scan_ids.append(scan_id)
    labels.append(label)

df = pd.DataFrame({
    "scan_id": scan_ids,
    "label": labels
})

print("Scan-level distribution:")
print(df["label"].value_counts())


#Splitting

In [ ]:
# ---- STEP 2: split (train 80 / temp 20) ----
train_df, temp_df = train_test_split(
    df,
    test_size=0.2,
    stratify=df["label"],
    random_state=42
)

In [ ]:
# ---- STEP 3: split temp → val/test (10/10) ----
val_df, test_df = train_test_split(
    temp_df,
    test_size=0.5,
    stratify=temp_df["label"],
    random_state=42
)

In [3]:
print("\nTrain distribution:")
print(train_df["label"].value_counts())

print("\nVal distribution:")
print(val_df["label"].value_counts())

print("\nTest distribution:")
print(test_df["label"].value_counts())

# ---- STEP 4: save ----
train_df.to_csv(os.path.join(output_dir, "train_scans.csv"), index=False)
val_df.to_csv(os.path.join(output_dir, "val_scans.csv"), index=False)
test_df.to_csv(os.path.join(output_dir, "test_scans.csv"), index=False)

print("\nSaved split files in:", output_dir)

Scan-level distribution:
label
0    39
1    36
Name: count, dtype: int64

Train distribution:
label
0    31
1    29
Name: count, dtype: int64

Val distribution:
label
0    4
1    3
Name: count, dtype: int64

Test distribution:
label
1    4
0    4
Name: count, dtype: int64

Saved split files in: /content/splits


#Save the CSV files

In [21]:
from google.colab import drive
import os

# mount drive
drive.mount('/content/drive')

# destination folder
drive_output_dir = "/content/drive/MyDrive/brain_ct_project/splits"
os.makedirs(drive_output_dir, exist_ok=True)

# save CSVs to Drive
train_df.to_csv(os.path.join(drive_output_dir, "train_scans.csv"), index=False)
val_df.to_csv(os.path.join(drive_output_dir, "val_scans.csv"), index=False)
test_df.to_csv(os.path.join(drive_output_dir, "test_scans.csv"), index=False)

print("✅ CSV files saved to:", drive_output_dir)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ CSV files saved to: /content/drive/MyDrive/brain_ct_project/splits
